# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## 1. Una recta para datos de fertilizante
Datos simulados: X es fertilizante (kg/ha), y es rendimiento (t/ha). `fit` aprende la recta; `predict` estima valores. Los puntos son observaciones y la línea es el ajuste. Esta primera figura usa entrenamiento; todavía no mide generalización. La pendiente describe asociación en el modelo, no una dosis causal recomendada.

In [ ]:
# ============================================
# Ejemplo simple de Regresión Lineal en Google Colab
# Contexto: Agronegocios
# ============================================

# -----------------------
# 1) IMPORTAR LIBRERÍAS
# -----------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# -----------------------
# 2) CREAR DATOS DE EJEMPLO
# -----------------------
# Relación: "Cantidad de fertilizante (kg/ha)" → "Rendimiento de maíz (ton/ha)"
X = np.array([50, 60, 70, 80, 90, 100, 110, 120, 130, 140]).reshape(-1, 1)
y = np.array([2.1, 2.4, 2.8, 3.1, 3.5, 3.7, 4.0, 4.3, 4.6, 4.8])

# -----------------------
# 3) AJUSTAR MODELO LINEAL
# -----------------------
modelo = LinearRegression()
modelo.fit(X, y)

# Predicciones
y_pred = modelo.predict(X)

# -----------------------
# 4) VISUALIZACIÓN
# -----------------------
plt.scatter(X, y, color="green", label="Datos reales (maíz)")
plt.plot(X, y_pred, color="red", linewidth=2, label="Regresión lineal")
plt.xlabel("Fertilizante aplicado (kg/ha)")
plt.ylabel("Rendimiento (ton/ha)")
plt.title("Regresión Lineal: Fertilizante vs Rendimiento de Maíz")
plt.legend()
plt.show()

# -----------------------
# 5) RESULTADOS
#Datos: simulamos una finca donde se mide el rendimiento del maíz en función del fertilizante aplicado.
#Relación: a mayor fertilización, mayor rendimiento (con cierta tendencia lineal).
#Modelo: ajusta una recta que predice la producción esperada de maíz dado un nivel de fertilización.
#Interpretación: la pendiente indica cuántas toneladas de maíz adicionales se obtienen por cada kilo extra de fertilizante por hectárea.
# -----------------------

print("Pendiente (coeficiente):", modelo.coef_[0])
print("Intercepto:", modelo.intercept_)

## 2. Una curva para una relación diferente
Este bloque cambia los datos: no es una comparación con la recta anterior sobre el mismo conjunto. El polinomio añade potencias de X. Lee la curva dentro del rango observado; un máximo del ejemplo no constituye una recomendación agronómica.

In [ ]:
# ============================================
# Ejemplo de Regresión Polinómica en Google Colab
# Contexto: Agronegocios
# ============================================

# -----------------------
# 1) IMPORTAR LIBRERÍAS
# -----------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# -----------------------
# 2) CREAR DATOS DE EJEMPLO
# -----------------------
# Relación: "Cantidad de fertilizante (kg/ha)" → "Rendimiento de maíz (ton/ha)"
# En este caso, después de cierto punto, aplicar más fertilizante reduce el rendimiento
X = np.array([50, 60, 70, 80, 90, 100, 110, 120, 130, 140]).reshape(-1, 1)
y = np.array([2.1, 2.7, 3.5, 4.2, 4.8, 5.0, 4.9, 4.6, 4.0, 3.5])

# -----------------------
# 3) TRANSFORMAR VARIABLES (POLINOMIO)
# -----------------------
poly = PolynomialFeatures(degree=2)  # cuadrático
X_poly = poly.fit_transform(X)

# Ajustar modelo polinómico
modelo_poly = LinearRegression()
modelo_poly.fit(X_poly, y)

# Predicciones
y_pred = modelo_poly.predict(X_poly)

# -----------------------
# 4) VISUALIZACIÓN
# -----------------------
plt.scatter(X, y, color="green", label="Datos reales (maíz)")
plt.plot(X, y_pred, color="red", linewidth=2, label="Regresión polinómica (grado 2)")
plt.xlabel("Fertilizante aplicado (kg/ha)")
plt.ylabel("Rendimiento (ton/ha)")
plt.title("Regresión Polinómica: Fertilizante vs Rendimiento de Maíz")
plt.legend()
plt.show()

# -----------------------
# 5) RESULTADOS
# Contexto: en agronegocios, aumentar fertilizante mejora el rendimiento solo hasta cierto punto; luego el exceso daña el cultivo.
# Polinómica: la curva (parábola) captura esa relación no lineal: crecimiento inicial, máximo, y caída posterior.
# Interpretación: este modelo ayuda a estimar la dosis óptima de fertilizante que maximiza el rendimiento, en lugar de asumir que “más siempre es mejor”.
# -----------------------
print("Coeficientes:", modelo_poly.coef_)
print("Intercepto:", modelo_poly.intercept_)


## 3. Controlar complejidad con Ridge y Lasso
Ahora usamos fertilizante y lluvia. La misma partición permite comparar OLS, Ridge y Lasso. Mira RMSE en entrenamiento y prueba; la regularización no gana por definición. En la curva parcial la lluvia se mantiene fija, mientras los puntos tienen lluvias diferentes.

In [ ]:
# ============================================================
# Regresión con Regularización: Ridge y Lasso (evitar sobreajuste)
# Contexto: Agronegocios
# Objetivo: predecir rendimiento de maíz (ton/ha) usando fertilizante y clima.
# ============================================================

# 1) IMPORTS
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 2) DATOS (sintéticos pero verosímiles)
# Variables base:
#   x1 = fertilizante (kg/ha)
#   x2 = lluvia en ciclo (mm)
# Intención: relación no lineal suave + ruido -> ideal para mostrar sobreajuste.
rng = np.random.RandomState(42)
n = 60
x1 = rng.uniform(60, 180, size=n)                 # fertilizante
x2 = rng.uniform(200, 800, size=n)                # lluvia
# Función "real" subyacente (desconocida para el modelo):
y_true = (
    0.02*x1 + 0.004*x2
    - 0.00006*(x1**2)                             # penaliza exceso de fertilizante
    + 0.0000005*(x2**2)                           # lluvia muy alta puede ayudar algo
    + 0.00003*x1*x2                               # interacción
)
y = y_true + rng.normal(0, 0.3, size=n)           # ruido
X = np.c_[x1, x2]

# 3) TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

# 4) DEFINICIÓN DE PIPELINES
# Nota clave: usamos PolynomialFeatures para generar muuuuchas columnas (hasta grado 5)
# Esto induce sobreajuste si no regularizamos.
poly_degree = 5

ols = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),  # con polinomios es más estable
    ("linreg", LinearRegression())
])

ridge = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),
    ("ridge", Ridge(alpha=5.0, random_state=0))   # alpha controla la fuerza de L2
])

lasso = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),
    ("lasso", Lasso(alpha=0.02, max_iter=50000, random_state=0))  # L1 puede poner coef=0
])

# 5) ENTRENAR
ols.fit(X_train, y_train)
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

# 6) EVALUACIÓN
def eval_model(name, pipe):
    yhat_tr = pipe.predict(X_train)
    yhat_te = pipe.predict(X_test)
    rmse_train = mean_squared_error(y_train, yhat_tr) ** 0.5
    rmse_test = mean_squared_error(y_test, yhat_te) ** 0.5
    return {
        "name": name,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "r2_train": r2_score(y_train, yhat_tr),
        "r2_test": r2_score(y_test, yhat_te),
        "n_coefs_nonzero": int(np.sum(
            np.abs(pipe.named_steps[list(pipe.named_steps.keys())[-1]].coef_) > 1e-8
        ))
    }
results = [
    eval_model("OLS (sin regularización)", ols),
    eval_model("Ridge (L2)", ridge),
    eval_model("Lasso (L1)", lasso),
]

for r in results:
    print(r["name"])
    print(f"  RMSE train: {r['rmse_train']:.3f} | RMSE test: {r['rmse_test']:.3f}")
    print(f"  R2   train: {r['r2_train']:.3f} | R2   test: {r['r2_test']:.3f}")
    print(f"  #coeficientes ≠ 0: {r['n_coefs_nonzero']}")
    print("-"*60)

# 7) VISUAL: curva parcial vs fertilizante manteniendo lluvia en su mediana
fert_grid = np.linspace(X[:,0].min(), X[:,0].max(), 120)
lluvia_med = np.median(X[:,1])
X_plot = np.c_[fert_grid, np.full_like(fert_grid, lluvia_med)]

plt.figure(figsize=(6,4))
plt.scatter(X_train[:,0], y_train, s=18, alpha=0.6, label="Train")
plt.scatter(X_test[:,0],  y_test,  s=18, alpha=0.6, label="Test")
plt.plot(fert_grid, ols.predict(X_plot),   lw=2, label="OLS")
plt.plot(fert_grid, ridge.predict(X_plot), lw=2, label="Ridge")
plt.plot(fert_grid, lasso.predict(X_plot), lw=2, label="Lasso")
plt.xlabel("Fertilizante (kg/ha)")
plt.ylabel("Rendimiento estimado (ton/ha)")
plt.title(f"Modelo parcial vs fertilizante (lluvia={lluvia_med:.0f} mm)")
plt.legend()
plt.tight_layout()
plt.show()

# 8) RESULTADOS (explicación práctica)
print("\nInterpretación práctica:")
print("""
- OLS usa todos los términos polinómicos sin límite: aprende el ruido y suele lucir
  excelente en TRAIN pero falla en TEST. Eso es sobreajuste.
- Ridge (L2) encoge todos los coeficientes hacia 0 de forma suave: reduce varianza
  y mejora la generalización (menor brecha train→test). Útil cuando todos los
  predictores aportan algo pero queremos controlar su magnitud.
- Lasso (L1) además de encoger, puede poner coeficientes EXACTAMENTE en 0:
  hace selección de variables automática. Útil cuando sospechas que muchas
  combinaciones polinómicas son irrelevantes.
- En agronegocios esto se traduce en recomendaciones más estables sobre dosis de
  fertilizante y respuesta bajo lluvia típica; la regularización evita
  "recetas" demasiado sensibles a mediciones accidentales o a una campaña atípica.
""")
